# Clean FoodHub Products CSV

Notebook này dùng để làm sạch file `foodhub_products_clean.csv` theo yêu cầu:

- `price`: chỉ giữ **1 giá**, bỏ ký tự `đ`, dấu `.`, và chuyển thành số nguyên.
- Thêm `voucher`: số phần trăm ngẫu nhiên là bội của 5, có thể là 0 và nhỏ hơn 30.
- Thêm `distance_km`: khoảng cách giả lập theo km.
- Thêm `prep_time_minutes`: thời gian lên món dự kiến theo phút.
- Thêm `rating`: từ 3.0 đến 4.9, làm tròn 1 chữ số sau dấu phẩy.
- `sold_count`: fake số nguyên dương ngẫu nhiên từ 50 đến 1500.
- Giữ lại các trường gốc: `url`, `name`, `price`, `ingredients`, `origin`, `product_info`, `description`, `category`, `product_code`, `brand`, `sold_count`, `image_urls`, `error`.

> Đặt file notebook này cùng thư mục với `foodhub_products_clean.csv`, sau đó chạy lần lượt từng cell.


In [1]:
import os
import re
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

# =========================
# Cấu hình đường dẫn
# =========================
INPUT_PATH = Path("foodhub_products_clean.csv")
OUTPUT_PATH = Path("foodhub_products_cleaned.csv")

# Nếu bạn muốn kết quả random có thể tái lập, giữ seed này.
# Nếu muốn mỗi lần chạy ra dữ liệu fake khác nhau, đổi thành None.
RANDOM_SEED = 42

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)


/home/laseb/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Input: foodhub_products_clean.csv
Output: foodhub_products_cleaned.csv


## 1. Đọc dữ liệu

Cell này ưu tiên đọc `foodhub_products_clean.csv`. Nếu bạn đang có file JSON thay vì CSV, đổi `INPUT_PATH` sang file `.json` hoặc đặt file JSON cùng tên `foodhub_products_clean.json`.


In [2]:
def read_foodhub_file(path: Path) -> pd.DataFrame:
    """Đọc file CSV hoặc JSON thành DataFrame."""
    if not path.exists():
        json_fallback = path.with_suffix(".json")
        if json_fallback.exists():
            print(f"Không thấy {path}, chuyển sang đọc {json_fallback}")
            path = json_fallback
        else:
            raise FileNotFoundError(
                f"Không tìm thấy {path} hoặc {json_fallback}. "
                "Hãy đặt file dữ liệu cùng thư mục với notebook."
            )

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    if path.suffix.lower() == ".json":
        # Hỗ trợ JSON dạng list object hoặc JSON lines.
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.read_json(path, lines=True)

    raise ValueError("Chỉ hỗ trợ file .csv hoặc .json")


df = read_foodhub_file(INPUT_PATH)
print("Số dòng:", len(df))
print("Số cột:", len(df.columns))
display(df.head())


Số dòng: 388
Số cột: 13


,url,name,price,ingredients,origin,product_info,description,category,product_code,brand,sold_count,image_urls,error
0,https://www.foodhub.vn/592682-bo-xao-mang-tay-...,Bò Xào Măng Tây (3-4 Người Ăn),99.000đ ~ 104.000đ,"[""Bước 1: Ướp ½ gói xốt xào với thịt bò."", ""Bư...","[""Bò không sử dụng chất tạo nạc, chất kích thí...",Măng tây xào thịt bò là món ngon nức tiếng đượ...,"Nguyên liệu được sơ chế sẵn, làm sạch, b...",Món Chiên Xào,8100897,FoodHub Kitchen,999,"[""https://cdn.abphotos.link/photos/resized/102...",NaN
1,https://www.foodhub.vn/561361-set-canh-cai-thi...,Canh Cải Thịt Bằm (4 Người Ăn),65.000đ ~ 75.000đ,"[""Nước cốt xương hầm lợn Quế: 200ml"", ""Cải ...","[""Được nuôi bằng giun quế với hàng lượng prote...","Canh Cải Thịt Bằm, món canh thanh mát, nhẹ vị,...","Canh Cải Thịt Bằm, món canh thanh mát, nhẹ vị,...",Món Canh,8100069,FoodHub Kitchen,2,"[""https://cdn.abphotos.link/photos/resized/102...",NaN
2,https://www.foodhub.vn/595813-hoa-cuc-vang-bo,Hoa Cúc Vàng (Bó),40.000đ,[],[],Hoa Cúc vàng là loài hoa cúng đem lại nhiều ma...,Hoa Cúc vàng là loài hoa cúng đem lại nhiều ma...,ĐỒ CÚNG,8101054,NaN,-32,"[""https://cdn.abphotos.link/photos/resized/102...",NaN
3,https://www.foodhub.vn/581338-cua-dong-nguyen-...,Cua Đồng Nguyên Chất Bà Bích (200gr/Gói),55.000đ,[],[],Cua đồng xay nguyên chất Bà Bích là một trong ...,Cua đồng xay nguyên chất Bà Bích là một trong ...,"Thủy, Hải Sản",8100955,Bà Bích,3,"[""https://cdn.abphotos.link/photos/resized/102...",NaN
4,https://www.foodhub.vn/548205-ma-dui-ga-nuong-...,Má Đùi Gà Nướng BBQ (~400gr),65.000đ ~ 70.000đ,"[""Má đùi đã tẩm ướp.""]","[""Không chất tăng trọng."", ""Không cúm gia cầm....",Má Đùi Gà Nướng BBQ gồm 2 má đùi N21 tư...,"Má Đùi Gà Nướng BBQ thơm ngon, hấp dẫn, ...",Món Nướng,8100387,FoodHub Kitchen,999,"[""https://cdn.abphotos.link/photos/resized/102...",NaN


## 2. Hàm làm sạch dữ liệu

Quy tắc xử lý giá:

Ví dụ `"63.000đ ~ 68.000đ"` sẽ lấy giá đầu tiên là `63000`.

Lý do: yêu cầu là chỉ có 1 giá, nên với khoảng giá notebook chọn **giá đầu tiên / giá thấp hơn** để dữ liệu nhất quán.


In [3]:
EXPECTED_COLUMNS = [
    "url",
    "name",
    "price",
    "ingredients",
    "origin",
    "product_info",
    "description",
    "category",
    "product_code",
    "brand",
    "sold_count",
    "image_urls",
    "error",
]


def normalize_text(value):
    """Chuẩn hóa text: bỏ khoảng trắng thừa, giữ chuỗi rỗng nếu thiếu."""
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def clean_price_to_int(value):
    """
    Chuyển giá về số nguyên.

    Ví dụ:
    - "63.000đ ~ 68.000đ" -> 63000
    - "50.000đ" -> 50000
    - "63,000 đ" -> 63000
    - "" hoặc NaN -> NaN
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    if not text:
        return np.nan

    # Với giá dạng khoảng: lấy phần giá đầu tiên.
    # Hỗ trợ các dấu phân tách phổ biến: ~, -, –.
    first_part = re.split(r"\s*(?:~|–|-)\s*", text)[0]

    # Bỏ toàn bộ ký tự không phải số.
    digits = re.sub(r"\D", "", first_part)

    if digits == "":
        return np.nan

    return int(digits)


def parse_list_like(value):
    """
    Đưa dữ liệu dạng list hoặc string biểu diễn list về list Python.

    Ví dụ:
    - "[]" -> []
    - "['a', 'b']" -> ['a', 'b']
    - "a, b" -> ['a', 'b']
    """
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    text = str(value).strip()
    if text == "" or text.lower() in {"nan", "none", "null"}:
        return []

    # Thử parse nếu là chuỗi dạng list Python.
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [normalize_text(x) for x in parsed if normalize_text(x)]
    except Exception:
        pass

    # Nếu không phải list, tách nhẹ theo dấu phẩy.
    if "," in text:
        return [normalize_text(x) for x in text.split(",") if normalize_text(x)]

    return [normalize_text(text)]


def extract_ingredients_from_description(description):
    """
    Cố gắng trích nguyên liệu từ description nếu cột ingredients đang rỗng.

    Hàm này chỉ là best-effort vì mô tả crawl có thể không có dấu xuống dòng hoặc dấu phẩy.
    """
    text = normalize_text(description)
    if not text:
        return []

    marker_match = re.search(r"\bNguyên liệu\b", text, flags=re.IGNORECASE)
    if not marker_match:
        return []

    after = text[marker_match.end():].strip()

    # Loại bỏ các marker thường gặp sau phần nguyên liệu nếu có.
    after = re.split(
        r"\b(?:Thông tin|Mô tả|Xuất xứ|Nguồn gốc|Hướng dẫn|Bảo quản)\b",
        after,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0].strip()

    if not after:
        return []

    # Nếu có dấu phân cách rõ ràng thì tách theo dấu đó.
    if any(sep in after for sep in [";", ",", "\\n", "•", "-"]):
        parts = re.split(r"[;,\\n•-]+", after)
        return [normalize_text(p) for p in parts if normalize_text(p)]

    # Fallback: dữ liệu không có dấu phân cách thì giữ nguyên đoạn sau "Nguyên liệu".
    return [after]


def clean_sold_count(value):
    """Chuyển sold_count về số nguyên, nếu thiếu thì để 0."""
    if pd.isna(value):
        return 0
    digits = re.sub(r"\D", "", str(value))
    return int(digits) if digits else 0


def make_voucher():
    """Voucher là bội của 5, có thể 0, nhỏ hơn 30."""
    return random.choice([0, 5, 10, 15, 20, 25])


def make_distance_km():
    """Fake khoảng cách từ 0.5 km đến 10.0 km."""
    return round(random.uniform(0.5, 10.0), 1)


def make_prep_time_minutes(distance_km):
    """
    Fake thời gian lên món dự kiến.
    Công thức đơn giản:
    - thời gian nấu/chuẩn bị cơ bản: 10-25 phút
    - cộng thêm khoảng 2 phút cho mỗi km
    """
    base_minutes = random.randint(10, 25)
    extra_minutes = int(round(distance_km * 2))
    return base_minutes + extra_minutes


def make_rating():
    """Fake rating từ 3.0 đến 4.9, làm tròn 1 chữ số sau dấu phẩy."""
    return round(random.uniform(3.0, 4.9), 1)


## 3. Làm sạch và thêm trường mới

In [4]:
# Tạo các cột thiếu nếu file hiện tại chưa đủ cột gốc.
for col in EXPECTED_COLUMNS:
    if col not in df.columns:
        df[col] = ""

# Chỉ giữ các cột gốc theo đúng thứ tự ban đầu.
clean_df = df[EXPECTED_COLUMNS].copy()

# Clean text columns.
text_columns = [
    "url",
    "name",
    "product_info",
    "description",
    "category",
    "product_code",
    "brand",
    "error",
]

for col in text_columns:
    clean_df[col] = clean_df[col].apply(normalize_text)

# Clean price: bỏ đ, bỏ dấu ., lấy 1 giá và chuyển sang int.
clean_df["price"] = clean_df["price"].apply(clean_price_to_int).astype("Int64")

# Clean list-like columns.
clean_df["ingredients"] = clean_df["ingredients"].apply(parse_list_like)
clean_df["origin"] = clean_df["origin"].apply(parse_list_like)
clean_df["image_urls"] = clean_df["image_urls"].apply(parse_list_like)

# Nếu ingredients rỗng, thử lấy từ description.
clean_df["ingredients"] = clean_df.apply(
    lambda row: row["ingredients"]
    if len(row["ingredients"]) > 0
    else extract_ingredients_from_description(row["description"]),
    axis=1,
)

# Fake sold_count: số nguyên dương ngẫu nhiên từ 50 đến 1500.
clean_df["sold_count"] = [random.randint(50, 1500) for _ in range(len(clean_df))]

# Thêm các trường fake.
clean_df["voucher"] = [make_voucher() for _ in range(len(clean_df))]
clean_df["distance_km"] = [make_distance_km() for _ in range(len(clean_df))]
clean_df["prep_time_minutes"] = clean_df["distance_km"].apply(make_prep_time_minutes).astype(int)
clean_df["rating"] = [make_rating() for _ in range(len(clean_df))]

# Sắp xếp lại thứ tự cột: cột gốc + cột mới.
FINAL_COLUMNS = EXPECTED_COLUMNS + [
    "voucher",
    "distance_km",
    "prep_time_minutes",
    "rating",
]
clean_df = clean_df[FINAL_COLUMNS]

display(clean_df.head())
print(clean_df.dtypes)


,url,name,price,ingredients,origin,product_info,description,category,product_code,brand,sold_count,image_urls,error,voucher,distance_km,prep_time_minutes,rating
0,https://www.foodhub.vn/592682-bo-xao-mang-tay-...,Bò Xào Măng Tây (3-4 Người Ăn),99000,"[Bước 1: Ướp ½ gói xốt xào với thịt bò., Bước ...","[Bò không sử dụng chất tạo nạc, chất kích thíc...",Măng tây xào thịt bò là món ngon nức tiếng đượ...,"Nguyên liệu được sơ chế sẵn, làm sạch, b...",Món Chiên Xào,8100897,FoodHub Kitchen,1359,[https://cdn.abphotos.link/photos/resized/1024...,,20,2.9,19,4.8
1,https://www.foodhub.vn/561361-set-canh-cai-thi...,Canh Cải Thịt Bằm (4 Người Ăn),65000,"[Nước cốt xương hầm lợn Quế: 200ml, Cải xan...",[Được nuôi bằng giun quế với hàng lượng protei...,"Canh Cải Thịt Bằm, món canh thanh mát, nhẹ vị,...","Canh Cải Thịt Bằm, món canh thanh mát, nhẹ vị,...",Món Canh,8100069,FoodHub Kitchen,278,[https://cdn.abphotos.link/photos/resized/1024...,,0,4.0,19,3.6
2,https://www.foodhub.vn/595813-hoa-cuc-vang-bo,Hoa Cúc Vàng (Bó),40000,"[khô, g tươi, go, ! Giao hà, g cực, ha, h tro,...",[],Hoa Cúc vàng là loài hoa cúng đem lại nhiều ma...,Hoa Cúc vàng là loài hoa cúng đem lại nhiều ma...,ĐỒ CÚNG,8101054,,101,[https://cdn.abphotos.link/photos/resized/1024...,,25,6.3,32,4.5
3,https://www.foodhub.vn/581338-cua-dong-nguyen-...,Cua Đồng Nguyên Chất Bà Bích (200gr/Gói),55000,[],[],Cua đồng xay nguyên chất Bà Bích là một trong ...,Cua đồng xay nguyên chất Bà Bích là một trong ...,"Thủy, Hải Sản",8100955,Bà Bích,613,[https://cdn.abphotos.link/photos/resized/1024...,,20,9.7,43,3.4
4,https://www.foodhub.vn/548205-ma-dui-ga-nuong-...,Má Đùi Gà Nướng BBQ (~400gr),65000,[Má đùi đã tẩm ướp.],"[Không chất tăng trọng., Không cúm gia cầm., H...",Má Đùi Gà Nướng BBQ gồm 2 má đùi N21 tư...,"Má Đùi Gà Nướng BBQ thơm ngon, hấp dẫn, ...",Món Nướng,8100387,FoodHub Kitchen,551,[https://cdn.abphotos.link/photos/resized/1024...,,10,7.3,26,4.9


url                      str
name                     str
price                  Int64
ingredients           object
origin                object
product_info             str
description              str
category                 str
product_code             str
brand                    str
sold_count             int64
image_urls            object
error                    str
voucher                int64
distance_km          float64
prep_time_minutes      int64
rating               float64
dtype: object


## 4. Kiểm tra nhanh dữ liệu sau khi clean

In [5]:
print("Số dòng sau khi clean:", len(clean_df))
print("\nSố giá bị thiếu:")
print(clean_df["price"].isna().sum())

print("\nKiểm tra voucher hợp lệ:")
valid_voucher = clean_df["voucher"].isin([0, 5, 10, 15, 20, 25]).all()
print(valid_voucher)

print("\nKiểm tra rating trong khoảng 3.0 đến 4.9:")
valid_rating = clean_df["rating"].between(3.0, 4.9).all()
print(valid_rating)

print("\nMô tả nhanh các cột số:")
display(clean_df[["price", "sold_count", "voucher", "distance_km", "prep_time_minutes", "rating"]].describe())


Số dòng sau khi clean: 388

Số giá bị thiếu:
0

Kiểm tra voucher hợp lệ:
True

Kiểm tra rating trong khoảng 3.0 đến 4.9:
True

Mô tả nhanh các cột số:


,price,sold_count,voucher,distance_km,prep_time_minutes,rating
count,388.0,388.000000,388.000000,388.000000,388.000000,388.000000
mean,68553.865979,760.786082,12.731959,5.178351,27.871134,3.918557
std,55195.616522,425.016191,8.483777,2.683984,7.063674,0.545042
min,200.0,51.000000,0.000000,0.500000,12.000000,3.000000
25%,21000.0,382.250000,5.000000,3.000000,23.000000,3.400000
50%,65000.0,743.000000,15.000000,5.000000,28.000000,3.900000
75%,95000.0,1151.500000,20.000000,7.600000,33.000000,4.400000
max,490000.0,1499.000000,25.000000,10.000000,43.000000,4.900000


## 5. Lưu file CSV mới

Do CSV không lưu được kiểu `list` thật của Python, các cột `ingredients`, `origin`, `image_urls` sẽ được lưu thành chuỗi JSON để đọc lại không bị lỗi.


In [6]:
save_df = clean_df.copy()

for col in ["ingredients", "origin", "image_urls"]:
    save_df[col] = save_df[col].apply(lambda x: json.dumps(x, ensure_ascii=False))

save_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Đã lưu file: {OUTPUT_PATH.resolve()}")
display(save_df.head())


Đã lưu file: /home/laseb/Documents/School/Vin  Aiiiii Learn/Day5/foodhub_output/foodhub_products_cleaned.csv


,url,name,price,ingredients,origin,product_info,description,category,product_code,brand,sold_count,image_urls,error,voucher,distance_km,prep_time_minutes,rating
0,https://www.foodhub.vn/592682-bo-xao-mang-tay-...,Bò Xào Măng Tây (3-4 Người Ăn),99000,"[""Bước 1: Ướp ½ gói xốt xào với thịt bò."", ""Bư...","[""Bò không sử dụng chất tạo nạc, chất kích thí...",Măng tây xào thịt bò là món ngon nức tiếng đượ...,"Nguyên liệu được sơ chế sẵn, làm sạch, b...",Món Chiên Xào,8100897,FoodHub Kitchen,1359,"[""https://cdn.abphotos.link/photos/resized/102...",,20,2.9,19,4.8
1,https://www.foodhub.vn/561361-set-canh-cai-thi...,Canh Cải Thịt Bằm (4 Người Ăn),65000,"[""Nước cốt xương hầm lợn Quế: 200ml"", ""Cải ...","[""Được nuôi bằng giun quế với hàng lượng prote...","Canh Cải Thịt Bằm, món canh thanh mát, nhẹ vị,...","Canh Cải Thịt Bằm, món canh thanh mát, nhẹ vị,...",Món Canh,8100069,FoodHub Kitchen,278,"[""https://cdn.abphotos.link/photos/resized/102...",,0,4.0,19,3.6
2,https://www.foodhub.vn/595813-hoa-cuc-vang-bo,Hoa Cúc Vàng (Bó),40000,"[""khô"", ""g tươi"", ""go"", ""! Giao hà"", ""g cực"", ...",[],Hoa Cúc vàng là loài hoa cúng đem lại nhiều ma...,Hoa Cúc vàng là loài hoa cúng đem lại nhiều ma...,ĐỒ CÚNG,8101054,,101,"[""https://cdn.abphotos.link/photos/resized/102...",,25,6.3,32,4.5
3,https://www.foodhub.vn/581338-cua-dong-nguyen-...,Cua Đồng Nguyên Chất Bà Bích (200gr/Gói),55000,[],[],Cua đồng xay nguyên chất Bà Bích là một trong ...,Cua đồng xay nguyên chất Bà Bích là một trong ...,"Thủy, Hải Sản",8100955,Bà Bích,613,"[""https://cdn.abphotos.link/photos/resized/102...",,20,9.7,43,3.4
4,https://www.foodhub.vn/548205-ma-dui-ga-nuong-...,Má Đùi Gà Nướng BBQ (~400gr),65000,"[""Má đùi đã tẩm ướp.""]","[""Không chất tăng trọng."", ""Không cúm gia cầm....",Má Đùi Gà Nướng BBQ gồm 2 má đùi N21 tư...,"Má Đùi Gà Nướng BBQ thơm ngon, hấp dẫn, ...",Món Nướng,8100387,FoodHub Kitchen,551,"[""https://cdn.abphotos.link/photos/resized/102...",,10,7.3,26,4.9


## 6. Đọc lại file để xác nhận

Cell này giúp kiểm tra file output đã được ghi ra thành công.


In [7]:
check_df = pd.read_csv(OUTPUT_PATH)
print("Số dòng đọc lại:", len(check_df))
display(check_df.head())


Số dòng đọc lại: 388


,url,name,price,ingredients,origin,product_info,description,category,product_code,brand,sold_count,image_urls,error,voucher,distance_km,prep_time_minutes,rating
0,https://www.foodhub.vn/592682-bo-xao-mang-tay-...,Bò Xào Măng Tây (3-4 Người Ăn),99000,"[""Bước 1: Ướp ½ gói xốt xào với thịt bò."", ""Bư...","[""Bò không sử dụng chất tạo nạc, chất kích thí...",Măng tây xào thịt bò là món ngon nức tiếng đượ...,"Nguyên liệu được sơ chế sẵn, làm sạch, b...",Món Chiên Xào,8100897,FoodHub Kitchen,1359,"[""https://cdn.abphotos.link/photos/resized/102...",NaN,20,2.9,19,4.8
1,https://www.foodhub.vn/561361-set-canh-cai-thi...,Canh Cải Thịt Bằm (4 Người Ăn),65000,"[""Nước cốt xương hầm lợn Quế: 200ml"", ""Cải ...","[""Được nuôi bằng giun quế với hàng lượng prote...","Canh Cải Thịt Bằm, món canh thanh mát, nhẹ vị,...","Canh Cải Thịt Bằm, món canh thanh mát, nhẹ vị,...",Món Canh,8100069,FoodHub Kitchen,278,"[""https://cdn.abphotos.link/photos/resized/102...",NaN,0,4.0,19,3.6
2,https://www.foodhub.vn/595813-hoa-cuc-vang-bo,Hoa Cúc Vàng (Bó),40000,"[""khô"", ""g tươi"", ""go"", ""! Giao hà"", ""g cực"", ...",[],Hoa Cúc vàng là loài hoa cúng đem lại nhiều ma...,Hoa Cúc vàng là loài hoa cúng đem lại nhiều ma...,ĐỒ CÚNG,8101054,NaN,101,"[""https://cdn.abphotos.link/photos/resized/102...",NaN,25,6.3,32,4.5
3,https://www.foodhub.vn/581338-cua-dong-nguyen-...,Cua Đồng Nguyên Chất Bà Bích (200gr/Gói),55000,[],[],Cua đồng xay nguyên chất Bà Bích là một trong ...,Cua đồng xay nguyên chất Bà Bích là một trong ...,"Thủy, Hải Sản",8100955,Bà Bích,613,"[""https://cdn.abphotos.link/photos/resized/102...",NaN,20,9.7,43,3.4
4,https://www.foodhub.vn/548205-ma-dui-ga-nuong-...,Má Đùi Gà Nướng BBQ (~400gr),65000,"[""Má đùi đã tẩm ướp.""]","[""Không chất tăng trọng."", ""Không cúm gia cầm....",Má Đùi Gà Nướng BBQ gồm 2 má đùi N21 tư...,"Má Đùi Gà Nướng BBQ thơm ngon, hấp dẫn, ...",Món Nướng,8100387,FoodHub Kitchen,551,"[""https://cdn.abphotos.link/photos/resized/102...",NaN,10,7.3,26,4.9
